# Building SQL AI Agent From Scratch

Source: [LinkedIn Learning - Build A SQL AI Agent for Production](https://www.linkedin.com/learning/build-with-ai-sql-ai-agents-in-production)

In [1]:
import sys
import os

current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.data import get_ibis_connection


In [2]:
tbl_name = "air_traffic"

postgres_config = {
    "user": "postgres",
    "password": "password",
    "host": "postgres",
    "port": 5432,
    "database": "my_db",
}

con = get_ibis_connection(
    backend="postgres",
    postgres_config=postgres_config,
)


In [3]:
from langchain_core.prompts import ChatPromptTemplate

system_template = """
Given the following SQL table, your job is to write queries given a user’s request.
Return just the SQL query as plain text, without additional text, and don't use markdown format. 
Please ensure that the field names in the query are enclosed in double quotes.
CREATE TABLE {tbl_name} ({schema})

""".strip()

user_template = "Write a SQL query that returns: {question}"

messages = [("system", system_template), ("user", user_template)]

prompt_template = ChatPromptTemplate.from_messages(messages)


In [4]:
from sql_ai_agent.db_handler import get_tbl_attr

tbl_attr = get_tbl_attr(con=con, tbl_name=tbl_name)

schema = tbl_attr.schema

print(schema)


Year bigint, Date timestamp without time zone, Operating Airline character varying, Operating Airline IATA Code character varying, Published Airline character varying, Published Airline IATA Code character varying, GEO Summary character varying, GEO Region character varying, Activity Type Code character varying, Price Category Code character varying, Terminal character varying, Boarding Area character varying, Passenger Count bigint


In [5]:
from langchain_openai import ChatOpenAI

base_url = "http://model-runner.docker.internal/engines/v1"
model = "gemma4:100k"

api_key = "docker"

llm = ChatOpenAI(
  base_url=base_url, 
  api_key=api_key, 
  temperature=0, model=model
  )

In [6]:
chain = prompt_template | llm

print(chain)

first=ChatPromptTemplate(input_variables=['question', 'schema', 'tbl_name'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['schema', 'tbl_name'], input_types={}, partial_variables={}, template="Given the following SQL table, your job is to write queries given a user’s request.\nReturn just the SQL query as plain text, without additional text, and don't use markdown format. \nPlease ensure that the field names in the query are enclosed in double quotes.\nCREATE TABLE {tbl_name} ({schema})"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Write a SQL query that returns: {question}'), additional_kwargs={})]) middle=[] last=ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0xffff3adccc90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletion

In [7]:
question = "How many passengers landed during 2024?"


In [8]:
llm_output = chain.invoke(
    {
        "question": question,
        "tbl_name": tbl_name,
        "schema": schema,
    }
)


In [9]:
query = llm_output.content
print(query)


SELECT SUM("Passenger Count") FROM air_traffic WHERE "Year" = 2024


In [10]:
con.sql(query).execute()


,sum
0,52210939


In [11]:
def basic_sql_agent(chain, question, tbl_name, schema, con):
    llm_output = chain.invoke(
        {
            "question": question,
            "tbl_name": tbl_name,
            "schema": schema,
        }
    )
    query = llm_output.content
    print("The return SQL query:")
    print("_" * 60)
    print(query)
    print("_" * 60)
    output = con.sql(query).execute()
    return output


In [12]:
basic_sql_agent(
    chain,
    question="how many passengers landed during 2024?",
    tbl_name=tbl_name,
    schema=schema,
    con=con,
)


The return SQL query:
____________________________________________________________
SELECT SUM("Passenger Count") FROM "air_traffic" WHERE "Year" = 2024
____________________________________________________________


,sum
0,52210939
